# 🏇 【未来レース予測 & 三連複特化】JRA出走表・オッズ対応 期待値AI競馬予想

本ノートブックは、過去データ（GitHub上のParquet）から学習した機械学習モデル（LightGBM）を用いて、**「週末・当日のJRA出走馬一覧JSON（出馬表＋最新オッズ）」** を入力するだけで、**未来のレースの三連複（3連複）推奨買い目と期待値を即座に出力**するGoogle Colab専用ノートブックです。

---

### 💡 なぜ「的中率」ではなく「期待値」なのか？
- 競馬には約20%〜25%の控除率が存在するため、人気馬ばかりを当てにいっても回収率は75%〜80%前後に収束します。
- 勝つためには、**「市場（オッズ）が過小評価している馬（真の複勝率 × オッズ > 1.0）」** をAIで発見し、期待値の高い三連複馬券を狙うバリューベッティング戦略が不可欠です。

---

### 📋 本ノートブックの使い方
1. **「ランタイム」 > 「すべてのセルを実行」** をクリックして、モデルの学習までを一括実行します（初回約30秒〜1分）。
2. ノートブック最下部の **「🎯 6. 未来レースの予想実行セル」** に、JRAの出走馬一覧JSONを貼り付けて実行すると、**全レースの三連複予想（◎○▲△☆印・軸1頭ながし・BOX）** が一瞬で出力されます！

## 1. 環境セットアップ & ライブラリのインストール

In [ ]:
# 必要なライブラリのインストール
%pip install -q pyarrow fastparquet lightgbm scikit-learn matplotlib seaborn tabulate

import os
import re
import json
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from tabulate import tabulate

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.unicode_minus'] = False
print('✅ ライブラリの準備が完了しました。')

## 2. GitHubから過去蓄積データの取得 & 統合

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/iinumac/keiba_prediction/main/data/processed'
RACES_URL = f'{BASE_URL}/races.parquet'
RESULTS_URL = f'{BASE_URL}/results.parquet'

print('📥 GitHubから過去レースデータを読み込み中...')
df_races = pd.read_parquet(RACES_URL)
df_results = pd.read_parquet(RESULTS_URL)

print(f'  - 過去レース数: {len(df_races):,} 件')
print(f'  - 過去出走馬レコード数: {len(df_results):,} 件')

# 過去データのマージ
overlap_cols = [c for c in df_races.columns if c in df_results.columns and c != 'race_id']
df_races_sub = df_races.drop(columns=overlap_cols)
df_history = pd.merge(df_results, df_races_sub, on='race_id', how='left')

print(f'📊 過去統合データ: {len(df_history):,} 行')
df_history.head(2)

## 3. 過去データの前処理 & 特徴量エンジニアリング

過去実績（着順、上がり3F、勝率、複勝率、騎手・調教師の押し上げ力 Added Value）を計算します。

In [ ]:
print('🧹 過去データの特徴量を生成中...')
clean_df = df_history.copy()

# 障害レース除外
if 'race_name' in clean_df.columns:
    clean_df = clean_df[~clean_df['race_name'].str.contains('障害', na=False)]
if 'surface' in clean_df.columns:
    clean_df = clean_df[~clean_df['surface'].astype(str).str.contains('障害', na=False)]

# 数値化
clean_df = clean_df[pd.to_numeric(clean_df['finish_position'], errors='coerce').notna()]
clean_df['finish_position'] = clean_df['finish_position'].astype(int)
clean_df['odds'] = pd.to_numeric(clean_df['odds'], errors='coerce')
clean_df['popularity'] = pd.to_numeric(clean_df['popularity'], errors='coerce')
clean_df = clean_df[clean_df['odds'].notna() & (clean_df['odds'] > 0)]

# 日付
clean_df['race_date'] = pd.to_datetime(clean_df['race_date'])
clean_df['year'] = clean_df['race_date'].dt.year

# 目的変数
clean_df['is_win'] = (clean_df['finish_position'] == 1).astype(int)
clean_df['is_top3'] = (clean_df['finish_position'] <= 3).astype(int)

# 馬・時系列ソート
clean_df = clean_df.sort_values(by=['horse_id', 'race_date'])

# 馬の過去実績（リークなし）
clean_df['horse_prev_win_rate'] = clean_df.groupby('horse_id')['is_win'].transform(lambda x: x.shift().expanding().mean()).fillna(0.08)
clean_df['horse_prev_top3_rate'] = clean_df.groupby('horse_id')['is_top3'].transform(lambda x: x.shift().expanding().mean()).fillna(0.24)
clean_df['prev_finish'] = clean_df.groupby('horse_id')['finish_position'].shift(1).fillna(10)
clean_df['prev_popularity'] = clean_df.groupby('horse_id')['popularity'].shift(1).fillna(10)
clean_df['prev_odds'] = clean_df.groupby('horse_id')['odds'].shift(1).fillna(30.0)
if 'last_3f' in clean_df.columns:
    clean_df['last_3f'] = pd.to_numeric(clean_df['last_3f'], errors='coerce')
    clean_df['prev_last_3f'] = clean_df.groupby('horse_id')['last_3f'].shift(1).fillna(36.5)
else:
    clean_df['prev_last_3f'] = 36.5
clean_df['prev_race_date'] = clean_df.groupby('horse_id')['race_date'].shift(1)
clean_df['days_since_last'] = (clean_df['race_date'] - clean_df['prev_race_date']).dt.days.fillna(90)
clean_df['is_debut'] = clean_df.groupby('horse_id').cumcount().apply(lambda x: 1 if x == 0 else 0)

# 騎手・調教師 Added Value
clean_df['jockey_diff'] = clean_df['is_top3'] - clean_df['horse_prev_top3_rate']
clean_df = clean_df.sort_values(by=['jockey_id', 'race_date'])
clean_df['jockey_added_value'] = clean_df.groupby('jockey_id')['jockey_diff'].transform(lambda x: x.shift().expanding().mean()).fillna(0.0)

clean_df['trainer_diff'] = clean_df['is_top3'] - clean_df['horse_prev_top3_rate']
clean_df = clean_df.sort_values(by=['trainer_id', 'race_date'])
clean_df['trainer_added_value'] = clean_df.groupby('trainer_id')['trainer_diff'].transform(lambda x: x.shift().expanding().mean()).fillna(0.0)

# 市場歪み特徴量
clean_df['market_implied_win_prob'] = 0.8 / clean_df['odds']
clean_df['min_odds_in_race'] = clean_df.groupby('race_id')['odds'].transform('min')
clean_df['odds_ratio_to_fav'] = clean_df['odds'] / (clean_df['min_odds_in_race'] + 1e-5)
clean_df['pop_odds_mismatch'] = clean_df['popularity'] * 2.5 - np.log(clean_df['odds'] + 1)

# コース
if 'surface' in clean_df.columns:
    clean_df['surface_code'] = clean_df['surface'].map({'芝': 0, 'ダート': 1, 'ダ': 1}).fillna(0)
else:
    clean_df['surface_code'] = 0

for col in ['impost', 'distance', 'horse_weight', 'horse_number', 'bracket_number']:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce').fillna(0)
    else:
        clean_df[col] = 0

# === 予測時に高速検索するための各馬・騎手・調教師の「最新マスタ辞書」を作成 ===
print('📦 未来レース参照用マスタを作成中...')
latest_horse_master = clean_df.sort_values('race_date').groupby('horse_name').tail(1).set_index('horse_name')
latest_jockey_master = clean_df.sort_values('race_date').groupby('jockey_name').tail(1).set_index('jockey_name')['jockey_added_value'].to_dict()
latest_trainer_master = clean_df.sort_values('race_date').groupby('trainer_name').tail(1).set_index('trainer_name')['trainer_added_value'].to_dict()

print(f'✅ 特徴量作成完了（レコード数: {len(clean_df):,} 行 / 登録馬数: {len(latest_horse_master):,} 頭）')

## 4. LightGBMによる3着内確率（複勝圏）予測モデルの学習

In [ ]:
FEATURE_COLS = [
    'popularity', 'odds', 'market_implied_win_prob', 'odds_ratio_to_fav', 'pop_odds_mismatch',
    'horse_prev_win_rate', 'horse_prev_top3_rate', 'prev_finish', 'prev_popularity', 'prev_odds', 'prev_last_3f', 'days_since_last', 'is_debut',
    'jockey_added_value', 'trainer_added_value',
    'distance', 'surface_code', 'impost', 'horse_weight', 'horse_number'
]
features = [f for f in FEATURE_COLS if f in clean_df.columns]
clean_df[features] = clean_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# 全データで学習
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

print('🚀 AIモデル（複勝圏内・3着内確率予測）を学習中...')
dtrain_top3 = lgb.Dataset(clean_df[features], label=clean_df['is_top3'])
model_top3 = lgb.train(lgb_params, dtrain_top3, num_boost_round=400)

print('🚀 AIモデル（1着勝率予測）を学習中...')
dtrain_win = lgb.Dataset(clean_df[features], label=clean_df['is_win'])
model_win = lgb.train(lgb_params, dtrain_win, num_boost_round=400)

print('✅ モデル学習が完了しました！いつでも未来レースを予想できます。')

## 5. JRA出走表JSONパーサー & 予想エンジンの定義

In [ ]:
def parse_jra_json_to_races(json_input):
    '''
    JRAの出走馬一覧JSON（文字列または辞書）から全レース情報をパースする
    '''
    if isinstance(json_input, str):
        try:
            json_input = json.loads(json_input)
        except Exception as e:
            print(f'JSONパースエラー: {e}')
            return []
            
    content = json_input.get('content', [])
    all_races = []
    race_counter = 1
    
    for elem in content:
        if elem.get('type') != 'list':
            continue
            
        for item in elem.get('items', []):
            text = item.get('text', '')
            if '発走時刻：' not in text or '単勝オッズ' not in text:
                continue
                
            # レース情報
            m_date = re.search(r'(\d{4})年(\d{1,2})月(\d{1,2})日', text)
            race_date = f'{m_date.group(1)}-{int(m_date.group(2)):02d}-{int(m_date.group(3)):02d}' if m_date else '2026-08-22'
            
            m_venue = re.search(r'\d+回([^\d\s]+)\d+日', text)
            venue = m_venue.group(1) if m_venue else '札幌'
            
            m_time = re.search(r'発走時刻：\s*(\d{1,2}時\d{1,2}分)', text)
            start_time = m_time.group(1) if m_time else ''
            
            m_course = re.search(r'コース：([\d,]+)\s*メートル\s*（(芝|ダート|ダ)\s*[・\s]*(右|左|直)?）', text)
            distance = int(m_course.group(1).replace(',', '')) if m_course else 1600
            surface = 'ダート' if (m_course and 'ダ' in m_course.group(2)) else '芝'
            surface_code = 1 if surface == 'ダート' else 0
            
            m_name = re.search(r'発走時刻：\s*\d{1,2}時\d{1,2}分\s*(\S+)', text)
            race_name = m_name.group(1) if m_name else f'第{race_counter}レース'
            
            # 出走馬トークン分割
            table_part = text.split('単勝オッズ')[-1].replace('ページトップへ戻る', '').strip()
            tokens = table_part.split()
            i = 0
            horses = []
            
            while i < len(tokens):
                token = tokens[i]
                if token.isdigit() or token in ['除外', '取消']:
                    num_str = token
                    if i + 1 >= len(tokens):
                        break
                    name = tokens[i+1]
                    i += 2
                    
                    weight = 470.0
                    is_debut = 0
                    if i < len(tokens) and tokens[i].isdigit() and i + 1 < len(tokens) and tokens[i+1] == 'kg':
                        weight = float(tokens[i])
                        i += 2
                        if i < len(tokens) and tokens[i].startswith('('):
                            i += 1
                    elif i < len(tokens) and '(初出走)' in tokens[i]:
                        is_debut = 1
                        i += 1
                        if i < len(tokens) and tokens[i].isdigit():
                            weight = float(tokens[i])
                            i += 1
                            if i < len(tokens) and tokens[i] == 'kg':
                                i += 1
                    elif i < len(tokens) and tokens[i].isdigit() and i + 2 < len(tokens) and tokens[i+1] == 'kg' and '(初出走)' in tokens[i+2]:
                        weight = float(tokens[i])
                        is_debut = 1
                        i += 3
                        
                    sex_age = ''
                    if i < len(tokens) and re.match(r'^(?:牡|牝|せん|セ)\d+$', tokens[i]):
                        sex_age = tokens[i]
                        i += 1
                        
                    impost = 55.0
                    if i < len(tokens) and re.match(r'^\d+(\.\d+)?$', tokens[i]):
                        impost = float(tokens[i])
                        i += 1
                        if i < len(tokens) and tokens[i] == 'kg':
                            i += 1
                            
                    if i < len(tokens) and tokens[i] in ['▲', '☆', '◇', '★']:
                        i += 1
                        
                    jt_tokens = []
                    odds = 0.0
                    is_excluded = False
                    while i < len(tokens):
                        if re.match(r'^\d+\.\d+$', tokens[i]):
                            odds = float(tokens[i])
                            i += 1
                            break
                        elif tokens[i] in ['除外', '取消']:
                            is_excluded = True
                            i += 1
                            break
                        else:
                            jt_tokens.append(tokens[i])
                            i += 1
                            
                    if not is_excluded and num_str.isdigit():
                        if len(jt_tokens) >= 4:
                            jockey = ''.join(jt_tokens[:2])
                            trainer = ''.join(jt_tokens[2:])
                        elif len(jt_tokens) == 2:
                            jockey = jt_tokens[0]
                            trainer = jt_tokens[1]
                        elif len(jt_tokens) == 3:
                            if '.' in jt_tokens[0]:
                                jockey = jt_tokens[0]
                                trainer = ''.join(jt_tokens[1:])
                            else:
                                jockey = ''.join(jt_tokens[:2])
                                trainer = jt_tokens[2]
                        else:
                            jockey = ' '.join(jt_tokens)
                            trainer = ''
                            
                        horses.append({
                            'horse_number': int(num_str),
                            'horse_name': name,
                            'jockey_name': jockey,
                            'trainer_name': trainer,
                            'impost': impost,
                            'horse_weight': weight,
                            'odds': odds,
                            'is_debut': is_debut,
                            'sex_age': sex_age
                        })
                else:
                    i += 1
                    
            if len(horses) > 0:
                all_races.append({
                    'race_num': race_counter,
                    'race_date': race_date,
                    'venue_name': venue,
                    'start_time': start_time,
                    'distance': distance,
                    'surface': surface,
                    'surface_code': surface_code,
                    'race_name': race_name,
                    'horses': pd.DataFrame(horses)
                })
                race_counter += 1
                
    return all_races


def predict_future_race(race_dict):
    '''
    出走馬表とオッズから過去実績を自動結合し、AI三連複予想を出力する
    '''
    rdf = race_dict['horses'].copy()
    if len(rdf) == 0:
        return
        
    # 人気順位の自動付与（オッズ昇順）
    rdf['popularity'] = rdf['odds'].rank(method='min').astype(int)
    rdf['distance'] = race_dict['distance']
    rdf['surface_code'] = race_dict['surface_code']
    
    # 過去データから馬実績をルックアップ
    horse_stats = []
    for _, row in rdf.iterrows():
        h_name = row['horse_name']
        if h_name in latest_horse_master.index:
            h_info = latest_horse_master.loc[h_name]
            if isinstance(h_info, pd.DataFrame):
                h_info = h_info.iloc[-1]
            p_win = h_info.get('horse_prev_win_rate', 0.08)
            p_top3 = h_info.get('horse_prev_top3_rate', 0.24)
            p_fin = h_info.get('prev_finish', 10)
            p_pop = h_info.get('prev_popularity', 10)
            p_odds = h_info.get('prev_odds', 30.0)
            p_l3f = h_info.get('prev_last_3f', 36.5)
            days = h_info.get('days_since_last', 60)
            debut = 0
        else:
            p_win, p_top3, p_fin, p_pop, p_odds, p_l3f, days, debut = 0.08, 0.20, 10, 10, 30.0, 36.5, 90, 1
            
        j_added = latest_jockey_master.get(row['jockey_name'], 0.0)
        t_added = latest_trainer_master.get(row['trainer_name'], 0.0)
        
        horse_stats.append({
            'horse_prev_win_rate': p_win,
            'horse_prev_top3_rate': p_top3,
            'prev_finish': p_fin,
            'prev_popularity': p_pop,
            'prev_odds': p_odds,
            'prev_last_3f': p_l3f,
            'days_since_last': days,
            'is_debut': 1 if row['is_debut'] == 1 else debut,
            'jockey_added_value': j_added,
            'trainer_added_value': t_added
        })
        
    stat_df = pd.DataFrame(horse_stats)
    for col in stat_df.columns:
        rdf[col] = stat_df[col].values
        
    # 市場歪み特徴量
    rdf['market_implied_win_prob'] = 0.8 / rdf['odds']
    min_o = rdf['odds'].min()
    rdf['odds_ratio_to_fav'] = rdf['odds'] / (min_o + 1e-5)
    rdf['pop_odds_mismatch'] = rdf['popularity'] * 2.5 - np.log(rdf['odds'] + 1)
    
    # 欠損補完 & 推論
    X = rdf[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    raw_top3 = model_top3.predict(X)
    raw_win = model_win.predict(X)
    
    # 確率のレース内正規化
    rdf['pred_top3_prob'] = np.clip(raw_top3 * (3.0 / (raw_top3.sum() + 1e-9)), 0.02, 0.98)
    rdf['pred_win_prob'] = raw_win / (raw_win.sum() + 1e-9)
    
    # 複勝推定オッズ & 期待値(EV)
    rdf['fukusho_odds_est'] = np.clip(1.0 + (rdf['odds'] - 1.0) * 0.28, 1.1, 30.0)
    rdf['ev_top3'] = rdf['pred_top3_prob'] * rdf['fukusho_odds_est']
    rdf['ev_win'] = rdf['pred_win_prob'] * rdf['odds']
    
    # 総合スコアと印の割り当て
    rdf['score'] = rdf['pred_top3_prob'] * 0.65 + (rdf['ev_top3'] / rdf['ev_top3'].max()) * 0.35
    rdf = rdf.sort_values(by='score', ascending=False).reset_index(drop=True)
    
    marks = ['◎ (本命)', '◯ (対抗)', '▲ (単穴)', '△ (連下)', '△ (連下)', '☆ (特注穴)']
    rdf['mark'] = [marks[i] if i < len(marks) else '―' for i in range(len(rdf))]
    
    # 出力ヘッダー
    r_num = race_dict.get('race_num', 1)
    r_name = race_dict['race_name']
    r_time = race_dict['start_time']
    r_date = race_dict['race_date']
    venue = race_dict['venue_name']
    course_str = f"{race_dict['surface']}{race_dict['distance']}m"
    
    print('=' * 75)
    print(f'🏇 【第{r_num}R {r_name}】 {r_date} {venue} {course_str} (発走: {r_time})')
    print('=' * 75)
    
    disp_df = rdf[['mark', 'horse_number', 'horse_name', 'sex_age', 'impost', 'jockey_name', 'popularity', 'odds', 'pred_top3_prob', 'ev_top3']].copy()
    disp_df['pred_top3_prob'] = (disp_df['pred_top3_prob'] * 100).map('{:.1f}%'.format)
    disp_df['ev_top3'] = disp_df['ev_top3'].map('{:.2f}'.format)
    disp_df['impost'] = disp_df['impost'].map('{:.1f}kg'.format)
    disp_df['odds'] = disp_df['odds'].map('{:.1f}'.format)
    
    print(tabulate(
        disp_df.values,
        headers=['印', '馬番', '馬名', '性齢', '斤量', '騎手', '人気', '単勝オッズ', 'AI複勝率', '複勝EV'],
        tablefmt='github'
    ))
    
    # 三連複 買い目提案
    axis = rdf.iloc[0]
    h_axis = int(axis['horse_number'])
    sub_opp = rdf.iloc[1:6]
    h_opps = [int(x) for x in sub_opp['horse_number'].tolist()]
    
    print('\n🎯 【AI推奨 三連複買い目】')
    print(f'  ① 【軸1頭ながし】 (10点)')
    print(f'     軸馬 (◎): [{h_axis:02d}] {axis["horse_name"]} (AI複勝率 {axis["pred_top3_prob"]*100:.1f}%, 期待値 {axis["ev_top3"]:.2f})')
    print(f'     相手 (◯▲△☆): {[f"{h:02d}" for h in h_opps]}')
    
    if len(rdf) >= 5:
        h_box = [int(x) for x in rdf.iloc[:5]['horse_number'].tolist()]
        print(f'  ② 【高期待値 5頭BOX】 (10点)')
        print(f'     買い目: {[f"{h:02d}" for h in h_box]}')
    print('=' * 75 + '\n')

print('✅ 予想エンジン定義完了')

## 🎯 6. 未来レースの予想実行セル（JSONを貼り付けて実行）

JRAの出走馬一覧ページから取得したJSONデータを以下の `jra_json_input` 変数に貼り付け、セルを実行するだけで全レースの三連複予想が表示されます。

In [ ]:
# === ここにJRA出走馬一覧のJSONを貼り付けて実行してください ===
jra_json_input = '''
{
  "url": "https://www.jra.go.jp/JRADB/accessD.html",
  "title": "出走馬一覧　JRA",
  "content": [
    {
      "type": "list",
      "items": [
        {
          "text": "2026年8月22日（土曜） 2回札幌1日 発走時刻： 10時00分 2歳未勝利 2歳 未勝利 （混合）［指定］ 馬齢 コース：1,700 メートル （ダート・右） 本賞金 （万円） 1着 590 2着 240 3着 150 4着 89 5着 59 枠 馬番 馬名 馬体重 性齢 負担重量 騎手名 調教師名 単勝オッズ 1 レイテスツドリーム 478 kg (-4) 牡2 52.0 kg ▲ 和田 陽希 平田 修 106.1 2 アンバーバレット 464 kg (+4) 牡2 54.0 kg ☆ 長浜 鴻緒 矢嶋 大樹 7.5 3 サーギュスターヴ 484 kg (+4) 牡2 55.0 kg 丹内 祐次 杉浦 宏昭 26.9 4 ファラオクィーン 458 kg (-2) 牝2 55.0 kg D.レーン 牧浦 充徳 2.7 5 アークレオン 464 kg (+2) 牡2 55.0 kg 小沢 大仁 高柳 大輔 263.0 6 マーズエクレール 520 kg (+6) 牡2 55.0 kg 鮫島 克駿 佐藤 悠太 33.9 7 トゥザファイナル 550 kg (+2) 牡2 55.0 kg C.ルメール 小島 茂之 1.8 8 マルカローゼ 468 kg (0) 牝2 55.0 kg R.ヘルナンデス 浜田 多実雄 170.2 9 シンゼンノト 480 kg (-6) 牡2 52.0 kg ▲ 遠藤 汰月 荒川 義之 63.2 10 レゾルートハート 484 kg (+6) 牡2 55.0 kg 岩田 望来 畑端 省吾 19.9 11 サウンドエフォート 494 kg (+14) 牡2 55.0 kg 吉田 隼人 安達 昭夫 20.2 12 ミライショウグン 496 kg (-4) 牡2 55.0 kg 石川 倭 前川 恭子 239.4 ページトップへ戻る"
        },
        {
          "text": "2026年8月22日（土曜） 2回札幌1日 発走時刻： 10時30分 2歳未勝利 2歳 未勝利 [指定] 馬齢 コース：1,800 メートル （芝・右） 本賞金 （万円） 1着 590 2着 240 3着 150 4着 89 5着 59 枠 馬番 馬名 馬体重 性齢 負担重量 騎手名 調教師名 単勝オッズ 1 ランダムナンバー 450 kg (+2) 牡2 55.0 kg 古川 吉洋 橋田 宜長 7.2 2 ランフォーザエイジ 452 kg (-4) 牝2 55.0 kg 浜中 俊 小島 茂之 1.5 3 ロードレスポンス 438 kg (+10) 牡2 55.0 kg Z.ロイド 和田 勇介 39.7 4 イントゥーザブルー 470 kg (0) 牡2 55.0 kg 丹内 祐次 清水 英克 9.5 5 ローレルメビウス 450 kg (+2) 牡2 53.0 kg ◇ 古川 奈穂 金成 貴史 72.6 6 ヒアカムズザサン 452 kg (+2) 牡2 55.0 kg 松山 弘平 中竹 和也 2.9 ページトップへ戻る"
        }
      ]
    }
  ]
}
'''

# パースの実行
parsed_races = parse_jra_json_to_races(jra_json_input)
print(f'🏁 検出されたレース数: {len(parsed_races)} レース\n')

# 全レースの三連複予想を出力
for r in parsed_races:
    predict_future_race(r)
